# **Get data**

In [ ]:
! mkdir ~/.kaggle

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download dmitrybabko/speech-emotion-recognition-en

In [ ]:
! unzip speech-emotion-recognition-en

# *Imports*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import librosa
import librosa.display
from IPython.display import Audio

from keras.callbacks import ModelCheckpoint
from keras.callbacks import LearningRateScheduler
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
from collections.abc import Iterable
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, BatchNormalization
from keras import layers, callbacks

# Data preparation

In [ ]:

crema_path = '/content/Crema/'
crema_dir_list = os.listdir(crema_path)

emotions_crema = []
paths_crema = []

for it in crema_dir_list:
    paths_crema.append(crema_path + it)

    part = it.split('_')
    if part[2] == 'SAD':
        emotions_crema.append('sad')
    elif part[2] == 'ANG':
        emotions_crema.append('angry')
    elif part[2] == 'DIS':
        emotions_crema.append('disgust')
    elif part[2] == 'FEA':
        emotions_crema.append('fear')
    elif part[2] == 'HAP':
        emotions_crema.append('happy')
    elif part[2] == 'NEU':
        emotions_crema.append('neutral')
    else:
        emotions_crema.append('Unknown')


emotions_crema_df = pd.DataFrame(emotions_crema, columns=['label'])
path_crema_df = pd.DataFrame(paths_crema, columns=['path'])
df = pd.concat([emotions_crema_df, path_crema_df], axis=1)

In [ ]:
df.head(4)

# PLOT 1

In [ ]:
data_for_plot = df['label']

sns.set(rc={'figure.figsize':(12,8)})
sns.set_style('darkgrid')
sns.histplot(data_for_plot, color='#4FAEB0')
plt.xlabel('Emotion', fontsize=20)
plt.ylabel('')
plt.title('Label distribution', fontsize=30)

# PLOT 2

In [ ]:
sample_rate=44100
audio_arrays = []

for i in df['path']:
    y, sr = librosa.load(i, sr=sample_rate)
    audio_arrays.append(y)

df['audio_to_array'] = audio_arrays

In [ ]:
data_for_plot = df['audio_to_array'].apply(lambda a: len(a)/sample_rate)
print(data_for_plot)
plt.figure(figsize=(16,10), dpi= 80)
sns.histplot(data_for_plot, fill=False, bins=60, kde=True, line_kws={'lw': 6}, color='#0095ff')
plt.title("Audio length", fontsize=30, fontweight=3)
plt.ylabel('')
plt.xlabel('Length in s', fontsize=20)
plt.xticks(size=18)
plt.yticks(size=18)
plt.show()

# Preprocessing

In [ ]:
frame_size = 1024
hop_lenght = 512


def amplitude_env(signal, frame_size, hop_length):
    amplitude_envelope = []
    for i in range (0, len(signal), hop_length):
        current_frame_amplitude_env = max(signal[i:i+frame_size])
        amplitude_envelope.append(current_frame_amplitude_env)

    return np.array(amplitude_envelope)


def fancy_amplitude_env (signal, frame_size, hop_length):
    return np.array([max(signal[i:i+frame_size]) for i in range(0, signal.size, hop_length)])

In [ ]:
def extract_mfcc(filename):
    y, sr = librosa.load(filename, duration = 3, offset = 0.5)
    mfcc = np.mean(librosa.feature.mfcc(y = y, sr = sr, n_mfcc = 40).T, axis = 0)
    return mfcc

In [ ]:
x_mfcc = df['path'].apply(lambda x: extract_mfcc(x))

In [ ]:
X = [x for x in x_mfcc]
X = np.expand_dims(np.array(X), -1)

enc = OneHotEncoder()
y = np.array(df['label']).reshape(-1, 1)
y = enc.fit_transform(y).toarray()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model LSTM

## Build model

In [ ]:
model = Sequential([
    LSTM(256, return_sequences=False, input_shape=(40,1)),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(6, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()
print(Iterable)

## Train

In [ ]:
checkpoint_filepath = 'best_model.keras'
model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1)


def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler_callback = LearningRateScheduler(scheduler)

history = model.fit(X_train, y_train, epochs=400, batch_size=64)

## Test

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)

# Вывод тестовой точности
print(f'Test accuracy is {test_accuracy:.4f}')
print(f'Test loss is {test_loss:.4f}')

# PLOT 3

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# PLOT 4

In [ ]:
plt.figure(figsize=(10, 6))
plt.figure(figsize=(12, 8))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()